# Optimize FEM's Experiment with Ax


In [190]:
import os
import re
from pathlib import Path

import pandas as pd
from ax.api.client import Client
from ax.api.configs import ChoiceParameterConfig, RangeParameterConfig

In [191]:
# 1. Initialize the Client.
client = Client(random_seed=42)

In [192]:
folder_path = Path(os.getcwd()) / "data"

In [193]:
exp = pd.read_csv(folder_path / "experiments.csv")
display(exp)


FileNotFoundError: [Errno 2] No such file or directory: 'c:\\Users\\admin\\Coding\\warisa-students\\nat_project\\src\\fem_exp_v2\\data\\experiments.csv'

In [ ]:
# Filter out rows with NaN values in the "deformation" column
filt = exp["deformation"].isna()
exp = exp[~filt]
exp

,trial_index,L_resin,L_metal,L_wood,deformation
0,0,5.215009,34.220961,60.564030,57.52
1,1,27.339656,22.836606,49.823738,64.63
2,2,40.027484,41.409247,18.563269,92.68
3,3,18.305284,3.095676,78.599040,32.65
4,4,0.292806,38.618832,61.088362,68.35
5,5,25.567594,2.093307,72.339099,38.43
6,6,11.265863,5.703480,83.030657,28.93
7,7,13.479070,0.000000,86.520930,32.45


In [ ]:
# 2. Configure where Ax will search.
client.configure_experiment(
    name="parn_exp",
    parameters=[
        RangeParameterConfig(
            name="L_resin",
            bounds=(0, 50),
            parameter_type="float",
        ),
        RangeParameterConfig(
            name="L_metal",
            bounds=(0, 50),
            parameter_type="float",
        ),
        # ChoiceParameterConfig(
        #     name="M",
        #     values=["Metal", "Resin", "Wood"],
        #     parameter_type="str",
        # ),
    ],
)

In [ ]:
client.configure_optimization(objective="-deformation")

In [ ]:
for idx, row in exp.iterrows():
    # trial_index,L_resin,L_metal,L_wood,deformation
    trial_index = client.attach_trial(
        parameters={
            "L_resin": row["L_resin"],
            "L_metal": row["L_metal"],
        }
    )
    client.complete_trial(
        trial_index=trial_index, raw_data={"deformation": float(row["deformation"])},
    )

[INFO 06-17 12:48:09] ax.api.client: Trial 0 marked COMPLETED.
[INFO 06-17 12:48:09] ax.api.client: Trial 1 marked COMPLETED.
[INFO 06-17 12:48:09] ax.api.client: Trial 2 marked COMPLETED.


[INFO 06-17 12:48:09] ax.api.client: Trial 3 marked COMPLETED.
[INFO 06-17 12:48:09] ax.api.client: Trial 4 marked COMPLETED.
[INFO 06-17 12:48:09] ax.api.client: Trial 5 marked COMPLETED.
[INFO 06-17 12:48:09] ax.api.client: Trial 6 marked COMPLETED.
[INFO 06-17 12:48:09] ax.api.client: Trial 7 marked COMPLETED.


In [ ]:
suggest_params = []
for i in range(3):
    data = client.get_next_trials(max_trials=1)
    for trial_index, parameters in data.items():
        # print(f"Trial {trial_index} with parameters {parameters}")
        suggest_params.append({"trial_index": trial_index, **parameters})
        # client.complete_trial(
        #     trial_index=trial_index,
        #     raw_data={},
        # )
_exp = pd.DataFrame(suggest_params)
_exp["L_wood"] = 100 - _exp["L_resin"] - _exp["L_metal"]
_exp["deformation"] = pd.NA
exp = pd.concat([exp, _exp], ignore_index=True).reset_index(drop=True)
exp

[INFO 06-17 12:48:09] ax.api.client: GenerationStrategy(name='Center+Sobol+MBM:fast', nodes=[CenterGenerationNode(next_node_name='Sobol'), GenerationNode(name='Sobol', generator_specs=[GeneratorSpec(generator_enum=Sobol, generator_key_override=None)], transition_criteria=[MinTrials(transition_to='MBM'), MinTrials(transition_to='MBM')], suggested_experiment_status=ExperimentStatus.INITIALIZATION, pausing_criteria=[MaxTrialsAwaitingData(threshold=5)]), GenerationNode(name='MBM', generator_specs=[GeneratorSpec(generator_enum=BoTorch, generator_key_override=None)], transition_criteria=None, suggested_experiment_status=ExperimentStatus.OPTIMIZATION, pausing_criteria=None)]) chosen based on user input and problem structure.
[INFO 06-17 12:48:09] ax.api.client: Generated new trial 8 with parameters {'L_resin': 25.0, 'L_metal': 25.0} using GenerationNode CenterOfSearchSpace.
[INFO 06-17 12:48:09] ax.api.client: Generated new trial 9 with parameters {'L_resin': 22.121445, 'L_metal': 46.7682} us

,trial_index,L_resin,L_metal,L_wood,deformation
0,0,5.215009,34.220961,60.564030,57.52
1,1,27.339656,22.836606,49.823738,64.63
2,2,40.027484,41.409247,18.563269,92.68
3,3,18.305284,3.095676,78.599040,32.65
4,4,0.292806,38.618832,61.088362,68.35
5,5,25.567594,2.093307,72.339099,38.43
6,6,11.265863,5.703480,83.030657,28.93
7,7,13.479070,0.000000,86.520930,32.45
8,8,25.000000,25.000000,50.000000,<NA>
9,9,22.121445,46.768200,31.110355,<NA>


In [ ]:
exp.to_excel(folder_path / "experiments.xlsx", index=False)